# 05a — Experiment Configuration

The **Experiment** class can be built from Python or from YAML. This notebook
shows both paths, how to inspect and override settings after creation, and
what the tunable parameters are.

## Creating from Python

In [1]:
from sklearn.linear_model import LogisticRegression
from skfair.experimentation import Experiment

exp = Experiment(
    datasets=["ricci"],
    methods=["Baseline", "FairSmote", "Massaging"],
    classifiers={"LogReg": LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)},
    metrics=["accuracy", "spd"],
    n_splits=2,
    random_state=42,
    audit_bias=False,
    audit_fairness=False,
)

After creation, every setting is a regular attribute you can inspect:

In [2]:
print("Datasets:  ", exp.dataset_names)
print("Methods:   ", exp.methods)
print("Classifiers:", list(exp.classifiers.keys()))
print("Metrics:   ", exp.metrics)
print("CV folds:  ", exp.n_splits)
print("Seed:      ", exp.random_state)
print("Bias audit:", exp.audit_bias)
print("Fair audit:", exp.audit_fairness)

Datasets:   ['ricci']
Methods:    ['Baseline', 'FairSmote', 'Massaging']
Classifiers: ['LogReg']
Metrics:    ['accuracy', 'spd']
CV folds:   2
Seed:       42
Bias audit: False
Fair audit: False


## Creating from YAML

The same experiment can be defined in a YAML file. This is useful for
reproducible benchmarks that you want to version-control separately from code.

You can also override `sens_attr` and `priv_group` directly on a dataset entry:
```yaml
datasets:
  - name: german
    sens_attr: age
    priv_group: 1
```

In [ ]:
with open("benchmark_config.yaml") as f:
    print(f.read())

In [ ]:
exp_cfg = Experiment.from_config("benchmark_config.yaml")

print("Datasets:  ", exp_cfg.dataset_names)
print("Methods:   ", exp_cfg.methods)
print("Classifiers:", list(exp_cfg.classifiers.keys()))
print("Metrics:   ", exp_cfg.metrics)
print("CV folds:  ", exp_cfg.n_splits)
print("Seed:      ", exp_cfg.random_state)
print("Bias audit:", exp_cfg.audit_bias)
print("Fair audit:", exp_cfg.audit_fairness)

## Overriding config settings from Python

Once created from YAML, the object is just a normal Experiment. You can
override any attribute before calling `run()`:

In [ ]:
# Narrow down to a quick test
exp_cfg.datasets = Experiment._resolve_datasets(["ricci"])
exp_cfg.methods = ["Baseline", "FairSmote"]
exp_cfg.n_splits = 2
exp_cfg.audit_bias = False
exp_cfg.audit_fairness = False

print("Datasets:  ", exp_cfg.dataset_names)
print("Methods:   ", exp_cfg.methods)
print("CV folds:  ", exp_cfg.n_splits)

## Available registries

The Experiment class draws datasets, methods, and metrics from built-in
registries. Here is everything available:

In [6]:
from skfair.experimentation import DATASET_REGISTRY, METHOD_REGISTRY, METRIC_REGISTRY

print("=== Datasets ===")
for name, info in DATASET_REGISTRY.items():
    print(f"  {name:20s}  sens_attr={info['sens_attr']!r}")

print("\n=== Methods ===")
for name, info in METHOD_REGISTRY.items():
    print(f"  {name:30s}  category={info['category']}")

print("\n=== Metrics ===")
for name, info in METRIC_REGISTRY.items():
    print(f"  {name:20s}  type={info['type']}")

=== Datasets ===
  adult                 sens_attr='sex'
  compas                sens_attr='sex'
  german                sens_attr='sex'
  heart_disease         sens_attr='sex'
  ricci                 sens_attr='Race'

=== Methods ===
  Baseline                        category=baseline
  Massaging                       category=sampler
  FairSmote                       category=sampler
  FairOversampling                category=sampler
  FAWOS                           category=sampler
  HeterogeneousFOS                category=sampler
  FairwayRemover                  category=sampler
  GeometricFairnessRepair         category=repair
  LearningFairRepresentations     category=repair
  ReweighingClassifier            category=meta
  FairBalanceClassifier           category=meta
  FairMask                        category=meta

=== Metrics ===
  accuracy              type=performance
  balanced_accuracy     type=performance
  disparate_impact      type=fairness
  spd                   ty

## Overriding dataset defaults

Registry datasets come with a default `sens_attr` and `priv_group` (shown above), but you can
override them per-experiment.

**Python API** — pass a `dataset_config` dict:

**YAML** — add `sens_attr` / `priv_group` on the dataset entry:
```yaml
datasets:
  - name: adult
    sens_attr: race
    priv_group: 1
```

You can also pass `method_config` to override method parameters.

In [7]:
exp_custom = Experiment(
    datasets=["ricci"],
    methods=["Baseline", "FairSmote"],
    metrics=["accuracy", "spd"],
    n_splits=2,
    random_state=42,
    dataset_config={
        "ricci": {"sens_attr": "Race", "priv_group": 1},
    },
    method_config={
        "FairSmote": {"random_state": 0, "k_neighbors": 3},
    },
)

print("Dataset config:", exp_custom.dataset_config)
print("Method config: ", exp_custom.method_config)

Dataset config: {'ricci': {'sens_attr': 'Race', 'priv_group': 1}}
Method config:  {'FairSmote': {'random_state': 0, 'k_neighbors': 3}}


## Auto-save and Load

The `Experiment` class supports automatic persistence. Pass `save_results`
and/or `save_object` to the constructor, and files are written automatically
at the end of `run()`. You can also call `save()` manually with overrides.

In [8]:
import os
from sklearn.linear_model import LogisticRegression
from skfair.experimentation import Experiment

os.makedirs("outputs", exist_ok=True)

# Auto-save: results CSV + full object pickle
exp_save = Experiment(
    datasets=["ricci"],
    methods=["Baseline", "FairSmote"],
    metrics=["accuracy", "spd"],
    n_splits=2,
    random_state=42,
    save_results=True,
    save_object=True,
    save_path="outputs/my_experiment",
)

results = exp_save.run(verbose=True)

# Check that files were created automatically
print("\nFiles created:")
for ext in (".csv", ".pkl"):
    fname = f"outputs/my_experiment{ext}"
    print(f"  {fname}: {os.path.exists(fname)}")

results


Dataset: Ricci
  Baseline                       | LogReg  acc=0.703  spd=-0.668


  FairSmote                      | LogReg  acc=0.788  spd=-0.065

Files created:
  outputs/my_experiment.csv: True
  outputs/my_experiment.pkl: True


,dataset,method,classifier,accuracy,spd
0,Ricci,Baseline,LogReg,0.703390,-0.667959
1,Ricci,FairSmote,LogReg,0.788136,-0.064761


### Loading a saved experiment

`Experiment.load()` restores the full object from a `.pkl` file — including
results, config, and any stored predictions or bias reports.

In [9]:
# Load the saved experiment
exp_loaded = Experiment.load("outputs/my_experiment.pkl")

print("Datasets:  ", exp_loaded.dataset_names)
print("Methods:   ", exp_loaded.methods)
print("Metrics:   ", exp_loaded.metrics)
print("Results shape:", exp_loaded.results_.shape)

# Generate a report from the loaded experiment
exp_loaded.to_report()

Datasets:   ['ricci']
Methods:    ['Baseline', 'FairSmote']
Metrics:    ['accuracy', 'spd']
Results shape: (2, 5)


### Manual save with overrides

`save()` also works as a standalone call. You can override the path and
choose which outputs to write:

In [10]:
# Manual save — only CSV, different path
exp_save.save("outputs/other_name", results=True, object=False)

print("outputs/other_name.csv exists:", os.path.exists("outputs/other_name.csv"))
print("outputs/other_name.pkl exists:", os.path.exists("outputs/other_name.pkl"))  # should be False

outputs/other_name.csv exists: True
outputs/other_name.pkl exists: False
